# 🏦 Loan Approval Prediction System

**Dataset:** [Loan Default — Kaggle](https://www.kaggle.com/datasets/nikhil1e9/loan-default)  
**Business Problem:** A bank wants to decide whether to approve a loan based on applicant attributes.

### Models Covered
| Model | Purpose |
|---|---|
| Logistic Regression | Loan Approved / Rejected |
| Random Forest | Risk Prediction |
| K-Means Clustering | Customer Segmentation |
| PCA | Dimensionality Reduction & Visualization |

## 1. Install & Import Libraries

In [ ]:
# Uncomment if running for the first time
# !pip install kagglehub imbalanced-learn scikit-learn pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('All libraries imported successfully ✅')

## 2. Load Dataset

In [ ]:
# Option A — Download from Kaggle
# import kagglehub
# path = kagglehub.dataset_download('nikhil1e9/loan-default')
# data = pd.read_csv(f'{path}/Loan_default.csv')

# Option B — Local file
data = pd.read_csv('Loan_default.csv')

print(f'Shape: {data.shape}')
data.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
data.info()

In [ ]:
data.describe().round(2)

In [ ]:
print('Missing values:')
print(data.isna().sum())

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = data['Default'].value_counts()
axes[0].bar(['No Default', 'Default'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(data)*100:.1f}%)',
                 ha='center', fontsize=10)

# Numeric distributions
num_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'InterestRate']
data[num_cols].hist(ax=axes[1], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_visible(False)  # hide placeholder
plt.tight_layout()
plt.show()

# Separate histogram grid
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), num_cols + ['MonthsEmployed']):
    ax.hist(data[col], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency')
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (after encoding)
data_enc = data.copy()
data_enc.drop(columns=['LoanID'], inplace=True, errors='ignore')
data_enc = pd.get_dummies(data_enc, drop_first=True)

corr = data_enc.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with Default
corr_target = data_enc.corr(numeric_only=True)['Default'].drop('Default').abs().sort_values(ascending=False)
print('Feature correlations with Default (absolute):')
print(corr_target.to_string())

## 4. Preprocessing

In [ ]:
IMPORTANT_FEATURES = [
    'Age', 'InterestRate', 'Income', 'MonthsEmployed',
    'LoanAmount', 'HasCoSigner_Yes', 'EmploymentType_Unemployed',
    'HasDependents_Yes', 'CreditScore'
]

# Ensure dummy columns exist
for col in ['HasCoSigner_Yes', 'EmploymentType_Unemployed', 'HasDependents_Yes']:
    if col not in data_enc.columns:
        data_enc[col] = 0

X = data_enc[IMPORTANT_FEATURES]
y = data_enc['Default']

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE — handle class imbalance
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'After SMOTE: {X_train_sm.shape}')
print(f'SMOTE class balance: {pd.Series(y_train_sm).value_counts().to_dict()}')

## 5. Logistic Regression — Loan Approved / Rejected

In [ ]:
lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sm, y_train_sm)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Precision : {precision_score(y_test, y_pred_lr)*100:.2f}%')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob_lr)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_lr, target_names=['No Default', 'Default']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['No Default', 'Default']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Logistic Regression', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
auc = roc_auc_score(y_test, y_prob_lr)
axes[1].plot(fpr, tpr, color='#3498db', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#3498db')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Random Forest — Risk Prediction

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_split=10,
    min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print('=== Random Forest ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Precision : {precision_score(y_test, y_pred_rf)*100:.2f}%')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob_rf)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_rf, target_names=['No Default', 'Default']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['No Default', 'Default']
).plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Confusion Matrix — Random Forest', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
auc = roc_auc_score(y_test, y_prob_rf)
axes[1].plot(fpr, tpr, color='#27ae60', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#27ae60')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Random Forest', fontweight='bold')
axes[1].legend()

# Feature importance
imp = pd.Series(rf.feature_importances_, index=IMPORTANT_FEATURES).sort_values()
colors = ['#27ae60' if i >= len(imp)-3 else '#a8d5b5' for i in range(len(imp))]
imp.plot(kind='barh', ax=axes[2], color=colors, edgecolor='white')
axes[2].set_title('Feature Importance', fontweight='bold')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 7. PCA — Dimensionality Reduction & Visualization

In [ ]:
X_scaled_all = scaler.transform(X)

# Fit full PCA
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled_all)

explained   = pca_full.explained_variance_ratio_
cumulative  = np.cumsum(explained)
n_components = len(explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, n_components+1), explained * 100,
            color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot', fontweight='bold')
axes[0].set_xticks(range(1, n_components+1))

# Cumulative variance
axes[1].plot(range(1, n_components+1), cumulative * 100,
             marker='o', color='#9b59b6', lw=2, markersize=7)
axes[1].axhline(90, color='#e74c3c', linestyle='--', lw=1.2, label='90% threshold')
axes[1].axhline(95, color='#e67e22', linestyle='--', lw=1.2, label='95% threshold')
axes[1].fill_between(range(1, n_components+1), cumulative * 100, alpha=0.1, color='#9b59b6')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xticks(range(1, n_components+1))
axes[1].legend()

plt.suptitle('PCA — Variance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
pca_summary = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(n_components)],
    'Explained Variance (%)': (explained * 100).round(2),
    'Cumulative (%)': (cumulative * 100).round(2)
})
print(pca_summary.to_string(index=False))

In [ ]:
# Biplot — PC1 vs PC2 with feature loadings
pca_2d = PCA(n_components=2, random_state=42)
X_pca  = pca_2d.fit_transform(X_scaled_all)

fig, ax = plt.subplots(figsize=(10, 7))

# Scatter colored by Default
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=y, cmap='RdYlGn_r', alpha=0.3, s=8, edgecolors='none'
)

# Feature loading arrows
loadings = pca_2d.components_.T
scale = 3.5
for i, feat in enumerate(IMPORTANT_FEATURES):
    ax.annotate('', xy=(loadings[i, 0]*scale, loadings[i, 1]*scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))
    ax.text(loadings[i, 0]*scale*1.12, loadings[i, 1]*scale*1.12,
            feat, fontsize=9, fontweight='bold', color='#2c3e50',
            ha='center', va='center')

plt.colorbar(scatter, ax=ax, label='Default (0 = No, 1 = Yes)')
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA Biplot — Feature Loadings + Default Labels', fontsize=13, fontweight='bold')
ax.axhline(0, color='gray', lw=0.5, linestyle='--')
ax.axvline(0, color='gray', lw=0.5, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Loading heatmap — which features contribute to which PCs
loadings_df = pd.DataFrame(
    pca_full.components_,
    columns=IMPORTANT_FEATURES,
    index=[f'PC{i+1}' for i in range(n_components)]
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('PCA Component Loadings Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. K-Means Clustering — Customer Segmentation

In [ ]:
# Elbow method
wcss = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled_all)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, wcss, marker='o', color='#e67e22', lw=2,
        markersize=8, markerfacecolor='#c0392b', markeredgewidth=0)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('WCSS (Inertia)')
ax.set_title('Elbow Method — Optimal k', fontweight='bold')
ax.set_xticks(K_range)
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled_all)

print('Cluster counts:')
print(pd.Series(clusters).value_counts().sort_index())

In [ ]:
# 2D PCA scatter coloured by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cluster labels
palette = ['#3498db', '#2ecc71', '#e74c3c']
for i, color in enumerate(palette):
    mask = clusters == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, alpha=0.5, s=10, label=f'Cluster {i+1}', edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('K-Means Clusters (PCA projection)', fontweight='bold')
axes[0].legend(markerscale=2)

# Default rate per cluster
cluster_df = X.copy()
cluster_df['Cluster'] = clusters + 1
cluster_df['Default'] = y.values

default_rate = cluster_df.groupby('Cluster')['Default'].mean() * 100
axes[1].bar(default_rate.index, default_rate.values,
            color=palette, edgecolor='white', width=0.5)
for i, v in enumerate(default_rate.values):
    axes[1].text(i+1, v+0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_title('Default Rate per Cluster', fontweight='bold')

plt.suptitle('K-Means Customer Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
profile = cluster_df.groupby('Cluster').mean().round(2)
print('Cluster Profiles (mean feature values):')
profile

## 9. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy (%)':  [
        round(accuracy_score(y_test, y_pred_lr)*100, 2),
        round(accuracy_score(y_test, y_pred_rf)*100, 2)
    ],
    'Precision (%)': [
        round(precision_score(y_test, y_pred_lr)*100, 2),
        round(precision_score(y_test, y_pred_rf)*100, 2)
    ],
    'ROC-AUC (%)': [
        round(roc_auc_score(y_test, y_prob_lr)*100, 2),
        round(roc_auc_score(y_test, y_prob_rf)*100, 2)
    ]
})
results.set_index('Model', inplace=True)
print(results.to_string())

# Visual comparison
fig, ax = plt.subplots(figsize=(9, 4))
results.plot(kind='bar', ax=ax, color=['#3498db','#27ae60','#9b59b6'],
             edgecolor='white', width=0.6)
ax.set_ylim(0, 110)
ax.set_ylabel('Score (%)')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax.set_xticklabels(results.index, rotation=0)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 10. Live Prediction — New Applicant

In [ ]:
# ── Edit the values below to test a new applicant ──
applicant = {
    'Age':                       35,
    'InterestRate':               8.5,
    'Income':                 60000,
    'MonthsEmployed':             48,
    'LoanAmount':             25000,
    'HasCoSigner_Yes':             0,   # 1 = Yes, 0 = No
    'EmploymentType_Unemployed':   0,   # 1 = Unemployed, 0 = Other
    'HasDependents_Yes':           1,   # 1 = Yes, 0 = No
    'CreditScore':               680,
}

df_input = pd.DataFrame([applicant])[IMPORTANT_FEATURES]
scaled_input = scaler.transform(df_input)

for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    pred = model.predict(scaled_input)[0]
    prob = model.predict_proba(scaled_input)[0]
    outcome = '✅ APPROVED' if pred == 0 else '❌ REJECTED'
    print(f'[{name}]  {outcome}  |  '
          f'P(No Default)={prob[0]*100:.1f}%  P(Default)={prob[1]*100:.1f}%')